In [1]:
import numpy as np
from scipy.signal import convolve2d
from HSIC import cka
import torch

In [2]:
def apply_conv_layer(images, kernel, mode='same'):
    return np.array([convolve2d(img, kernel, mode=mode, boundary='wrap')
                     for img in images])

def binarize(feature_maps, threshold=None):
    flat = feature_maps.reshape(len(feature_maps), -1)
    t = np.median(flat) if threshold is None else threshold
    return np.where(flat >= t, 1, -1).astype(np.int32)

In [3]:
def images_to_flat(imgs):
    arr = np.array(imgs, dtype=np.int32)
    assert arr.ndim == 3 and arr.shape[1:] == (4, 4), "Expected shape (N, 4, 4)"
    assert set(arr.flatten()).issubset({-1, 1}), "Images must be binary (-1/1)"
    return arr.reshape(len(arr), -1)

def empirical_pmf(patterns):
    keys = [tuple(row) for row in patterns]
    n = len(keys)
    counts = {}
    for k in keys:
        counts[k] = counts.get(k, 0) + 1
    return {k: v / n for k, v in counts.items()}

def entropy(pmf):
    return -sum(p * np.log2(p) for p in pmf.values() if p > 0)

def joint_pmf(pats_A, pats_B):
    assert len(pats_A) == len(pats_B)
    n = len(pats_A)
    jpdf = {}
    for a, b in zip(pats_A, pats_B):
        key = (tuple(a), tuple(b))
        jpdf[key] = jpdf.get(key, 0) + 1
    return {k: v / n for k, v in jpdf.items()}

def mutual_information(pmf_A, pmf_B, jpdf):
    mi = 0.0
    for (a, b), pab in jpdf.items():
        pa = pmf_A.get(a, 0)
        pb = pmf_B.get(b, 0)
        if pa > 0 and pb > 0 and pab > 0:
            mi += pab * np.log2(pab / (pa * pb))
    return mi

def renyi2_entropy(pmf):
    """Rényi-2 (collision) entropy:  H_2 = -log2( sum_x p(x)^2 )."""
    coll = sum(p * p for p in pmf.values())
    return -np.log2(coll) if coll > 0 else 0.0

def joint_renyi2_entropy(jpdf):
    """H_2(A,B) = -log2( sum_{a,b} p(a,b)^2 )."""
    coll = sum(p * p for p in jpdf.values())
    return -np.log2(coll) if coll > 0 else 0.0

def renyi2_mutual_information(pmf_A, pmf_B, jpdf):
    """
    'Naive' Rényi-2 MI via the entropy decomposition:
        I_2(A; B) = H_2(A) + H_2(B) - H_2(A, B)
    NOTE: unlike Shannon MI, this is NOT guaranteed >= 0 and can be
    slightly negative, since Rényi entropies aren't subadditive the
    same way Shannon entropy is.
    """
    H2_A  = renyi2_entropy(pmf_A)
    H2_B  = renyi2_entropy(pmf_B)
    H2_AB = joint_renyi2_entropy(jpdf)
    return H2_A + H2_B - H2_AB

def information_summary(flat_A, flat_B, label=""):
    pmf_A = empirical_pmf(flat_A)
    pmf_B = empirical_pmf(flat_B)
    jpdf  = joint_pmf(flat_A, flat_B)

    # --- Shannon ---
    H_A   = entropy(pmf_A)
    H_B   = entropy(pmf_B)
    MI    = mutual_information(pmf_A, pmf_B, jpdf)
    norm  = MI / max(H_A, H_B) if max(H_A, H_B) > 0 else 0.0

    # --- Rényi-2 (collision) ---
    H2_A     = renyi2_entropy(pmf_A)
    H2_B     = renyi2_entropy(pmf_B)
    MI2      = renyi2_mutual_information(pmf_A, pmf_B, jpdf)
    denom2   = max(H2_A, H2_B)
    norm2    = MI2 / denom2 if denom2 > 0 else 0.0

    print(f"\n{'─'*45}")
    print(f"  {label}")
    print(f"{'─'*45}")
    print(f"  [Shannon]")
    print(f"  H(A)            = {H_A:.4f} bits")
    print(f"  H(B)            = {H_B:.4f} bits")
    print(f"  MI(A; B)        = {MI:.4f} bits")
    print(f"  H(A|B)          = {H_A - MI:.4f} bits")
    print(f"  H(B|A)          = {H_B - MI:.4f} bits")
    print(f"  Normalized MI   = {norm:.4f}  (1.0 = deterministic)")
    print(f"  [Rényi-2 / collision]")
    print(f"  H2(A)           = {H2_A:.4f} bits")
    print(f"  H2(B)           = {H2_B:.4f} bits")
    print(f"  MI2(A; B)       = {MI2:.4f} bits")
    print(f"  Normalized MI2  = {norm2:.4f}")
    print(f"  CKA (Linear)    = {sum([cka(torch.tensor(flat_A[i:i+10000], dtype=torch.float32), torch.tensor(flat_B[i:i+10000], dtype=torch.float32), True, 'linear') for i in range(0, len(flat_A), 10000)])/7}")
    print(f"  CKA (RBF)       = {sum([cka(torch.tensor(flat_A[i:i+10000], dtype=torch.float32), torch.tensor(flat_B[i:i+10000], dtype=torch.float32), True, 'rbf') for i in range(0, len(flat_A), 10000)])/7}")
    return dict(
        H_A=H_A, H_B=H_B, MI=MI, norm_MI=norm,
        H2_A=H2_A, H2_B=H2_B, MI2=MI2, norm_MI2=norm2,
    )

In [4]:
KERNELS = {
    # "Identity"              : np.array([[0, 0, 0],
    #                                     [0, 1, 0],
    #                                     [0, 0, 0]], dtype=float),

    "Edge detect (Sobel-x)" : np.array([[-1, 0, 1],
                                        [-2, 0, 2],
                                        [-1, 0, 1]], dtype=float),

    # "Edge detect (Sobel-y)" : np.array([[-1,-2,-1],
    #                                     [ 0, 0, 0],
    #                                     [ 1, 2, 1]], dtype=float),

    # "Blur (box 3×3)"        : np.ones((3, 3), dtype=float) / 9,

    "Laplacian"             : np.array([[ 0,-1, 0],
                                        [-1, 4,-1],
                                        [ 0,-1, 0]], dtype=float),

    # "Random"                : rng.standard_normal((3, 3)),
}

In [5]:
import itertools

def generate_all_4x4_images():
    # Define the possible pixel values
    pixel_values = [-1, 1]
    
    # Generate all combinations for 16 pixels (4x4)
    all_combinations = itertools.product(pixel_values, repeat=16)
    
    # Convert combinations into a NumPy array and reshape to 4x4
    # Using float32 or int8 depending on your precision needs
    all_images = np.fromiter(itertools.chain(*all_combinations), dtype=np.int8).reshape(-1, 4, 4)
    
    return all_images

# Generate the images
base_images = generate_all_4x4_images()
set_A = base_images.astype(float)

for ROTATION_K in range(1, 4):
    print(f"Rotation: {ROTATION_K*90}°  |  Conv output binarized at shared median")
    set_B = np.rot90(base_images, k=ROTATION_K, axes=(-2, -1)).astype(float)

    # ── 1. Raw images (baseline) ───────────────────────────────────────────────────

    flat_A_raw = images_to_flat(set_A.astype(int))
    flat_B_raw = images_to_flat(set_B.astype(int))

    # ── 2. After each conv kernel ──────────────────────────────────────────────────

    for name, kernel in KERNELS.items():
        feat_A = apply_conv_layer(set_A, kernel, mode='same')
        feat_B = apply_conv_layer(set_B, kernel, mode='same')

        shared_threshold = np.median(np.concatenate([feat_A.ravel(), feat_B.ravel()]))
        q_A = binarize(feat_A, threshold=shared_threshold)
        q_B = binarize(feat_B, threshold=shared_threshold)

        res = information_summary(q_A, q_B, label=f"AFTER CONV — {name}")
    print("")

Rotation: 90°  |  Conv output binarized at shared median

─────────────────────────────────────────────
  AFTER CONV — Edge detect (Sobel-x)
─────────────────────────────────────────────
  [Shannon]
  H(A)            = 9.0427 bits
  H(B)            = 9.0427 bits
  MI(A; B)        = 4.6122 bits
  H(A|B)          = 4.4305 bits
  H(B|A)          = 4.4305 bits
  Normalized MI   = 0.5100  (1.0 = deterministic)
  [Rényi-2 / collision]
  H2(A)           = 8.1253 bits
  H2(B)           = 8.1253 bits
  MI2(A; B)       = 3.3184 bits
  Normalized MI2  = 0.4084
  CKA (Linear)    = 0.13748258352279663
  CKA (RBF)       = 0.13834784924983978

─────────────────────────────────────────────
  AFTER CONV — Laplacian
─────────────────────────────────────────────
  [Shannon]
  H(A)            = 15.1834 bits
  H(B)            = 15.1834 bits
  MI(A; B)        = 15.1834 bits
  H(A|B)          = 0.0000 bits
  H(B|A)          = 0.0000 bits
  Normalized MI   = 1.0000  (1.0 = deterministic)
  [Rényi-2 / collisio